In [ ]:
!which python
# try:
from scipy import constants as cons
import time
import scipy as sc
from matplotlib import gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy.fft as fft
import sys
# sys.path.append('/home/petronio/LPPview')
# sys.path.insert(0, '../LPPview/')
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import os

plt.rcParams["font.family"] = 'serif'

rcParams['axes.grid'] = True
plt.rcParams.update({'ytick.right': True, 'ytick.right': True})

class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout = self._original_stdout


def derivative(func, spacing):
    result = np.zeros_like(func)
    print(np.size(result), np.size(func))
    for i in range(1, np.size(func) - 1):
        result[i] = (func[i + 1] - func[i - 1]) / (2 * spacing)
    result[0] = result[1]
    result[-1] = result[-2]
    return result
colors = ["r", "orange", "darkgreen", "b", "black", "cyan"]
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=colors)

import string
listAlphabet = list(string.ascii_lowercase)


In [ ]:
def cumTrapz(y, dx_arr):
    n = y.shape[0]
    cuminteg = np.zeros(y.shape, dtype=float)
    cuminteg[0] = dx_arr[0] * y[0]
    for i in range(1, n):
        cuminteg[i] = cuminteg[i-1] + dx_arr[i] * y[i]

    return cuminteg


def compute_phi(fE, fDelta_x, fJ, fV, fRext):
    
    phi = fV - fJ * fRext - cumTrapz(fE, fDelta_x)  # Discharge electrostatic potential
    return phi

def gradient(y, x):
    dp_dz = np.zeros(y.shape)
    dp_dz[1:-1] = (y[2:] - y[:-2]) / (x[2:] - x[:-2])
    dp_dz[0] = 2 * dp_dz[1] - dp_dz[2]
    dp_dz[-1] = 2 * dp_dz[-2] - dp_dz[-3]

    return dp_dz

def compute_E_with_az_motion(fP, fB, fESTAR, wall_inter_type:str, fR1, fR2, fM, fx_center, fLTHR, fKEL, falpha_B, fJ, fA0, new_model=False):

    # TODO: This is already computed! Maybe move to the source
    #############################################################
    #       We give a name to the vars to make it more readable
    #############################################################
    ng = fP[0,:]
    ni = fP[1,:]
    ui = fP[2,:]
    Te = fP[3,:]
    ve = fP[4,:]
    Ue_y = fP[5,:]

    Gamma_i = ni*ui
    me = phy_const.m_e
    wce = phy_const.e*fB/me
    if new_model:

        div_p = gradient(phy_const.e*ni*Te, fx_center)  

        E = - Ue_y * fB - div_p / (phy_const.e * ni)
    else:
        #############################
        #       Compute the rates   #
        #############################

        sigma = 2.0 * Te / fESTAR  # SEE yield
        sigma[sigma > 0.986] = 0.986
        if wall_inter_type == "Default":
            # nu_iw value before Martin changed the code for Charoy's test cases.    
            nu_iw = (4./3.)*(1./(fR2 - fR1))*np.sqrt(phy_const.e*Te/fM)
            # Limit the wall interactions to the inner channel
            nu_iw[fx_center > fLTHR] = 0.0
            nu_ew = nu_iw / (1.0 - sigma)  # Electron - wall collision rate        
        
        elif wall_inter_type == "None":
            nu_iw = np.zeros(Te.shape, dtype=float)     # Ion - wall collision rate
            nu_ew = np.zeros(Te.shape, dtype=float)     # Electron - wall collision rate



        # TODO: Put decreasing wall collisions (Not needed for the moment)
        #    if decreasing_nu_iw:
        #        index_L1 = np.argmax(z > L1)
        #        index_LTHR = np.argmax(z > LTHR)
        #        index_ind = index_L1 - index_LTHR + 1
        #
        #        nu_iw[index_LTHR: index_L1] = nu_iw[index_LTHR] * np.arange(index_ind, 1, -1) / index_ind
        #        nu_iw[index_L1:] = 0.0

        ##################################################
        #       Compute the electron properties          #
        ##################################################

        nu_m = (
            ng * fKEL + falpha_B * wce + nu_ew
            )  # Electron momentum - transfer collision frequency
        
        mu_eff = (phy_const.e / (me* nu_m)) * (
            1.0 / (1 + (wce / nu_m) ** 2)
            )  # Effective mobility    dp_dz  = np.gradient(ni*Te, Delta_x)

        div_p = gradient(phy_const.e*ni*Te, fx_center)  

        E = - Ue_y * fB - phy_const.m_e / phy_const.e * nu_m * ve - div_p / (phy_const.e * ni)
    return E

def compute_E(fP):

    # TODO: This is already computed! Maybe move to the source
    #############################################################
    #       We give a name to the vars to make it more readable
    #############################################################
    ng = fP[0,:]
    ni = fP[1,:]
    Te = fP[3,:]
    ve = fP[4,:]

    me = phy_const.m_e
    wce     = phy_const.e*Barr/me   # electron cyclotron frequency
    
    #############################
    #       Compute the rates   #
    #############################

    sigma = 2.0 * Te / ESTAR  # SEE yield
    sigma[sigma > 0.986] = 0.986
    if wall_inter_type == "Default":
        # nu_iw value before Martin changed the code for Charoy's test cases.    
        nu_iw = (4./3.)*(1./(R2 - R1))*np.sqrt(phy_const.e*Te/Mi)
        # Limit the wall interactions to the inner channel
        nu_iw[x_center > LTHR] = 0.0
        nu_ew = nu_iw / (1.0 - sigma)  # Electron - wall collision rate        
    
    elif wall_inter_type == "None":
        nu_iw = np.zeros(Te.shape, dtype=float)     # Ion - wall collision rate
        nu_ew = np.zeros(Te.shape, dtype=float)     # Electron - wall collision rate

    # TODO: Put decreasing wall collisions (Not needed for the moment)
    #    if decreasing_nu_iw:
    #        index_L1 = np.argmax(z > L1)
    #        index_LTHR = np.argmax(z > LTHR)
    #        index_ind = index_L1 - index_LTHR + 1
    #
    #        nu_iw[index_LTHR: index_L1] = nu_iw[index_LTHR] * np.arange(index_ind, 1, -1) / index_ind
    #        nu_iw[index_L1:] = 0.0

    ##################################################
    #       Compute the electron properties          #
    ##################################################

    nu_m = (
        ng * KEL + alpha_B * wce + nu_ew
        )  # Electron momentum - transfer collision frequency
    
    mu_eff = (phy_const.e / (me* nu_m)) * (
        1.0 / (1 + (wce / nu_m) ** 2)
        )  # Effective mobility    dp_dz  = np.gradient(ni*Te, Delta_x)

    dp_dz  = gradient(ni*Te, x_center)
    
    E = - ve / mu_eff - dp_dz / ni  # Discharge electric field
    return E

In [ ]:
import pickle
import glob
import configparser
from scipy import constants as phy_const
import math
import os

##########################################################
#           POST-PROC PARAMETERS
##########################################################
homepath = "/home/petronio/runs/fluid_results/"

Results     = "../Results/test_neutrals/"
Results_1 = "/home/petronio/Nextcloud/code/FLEHET1D/alex_test_azim_motion/Results/new_results_1/"

# Results     = "no_heat_flux/"
label_res = Results[:3]

# Results = homepath + Results

# Results = "/home/petronio/Downloads/half_gradPxy_emp_term_add_3/"

image_folder= Results + "/Figs/"

ResultConfig = Results+'/Configuration.cfg'
# msp = SimuParameters(ResultConfig)

In [ ]:
##########################################################
#           Collects Large Unvariant Parameters
##########################################################
ResultsData = Results+"/Data"
ResultsData_1 = Results_1+"/Data"
try:
    with open(Results + "Data/MacroscopicUnvariants.pkl", 'rb') as f:
        [B_fluid, x_mesh, x_center, alpha_B] = pickle.load(f)
except:
    with open("/home/petronio/Nextcloud/code/FLEHET1D/Results/test_post_modifs_2/Data/MacroscopicUnvariants.pkl", 'rb') as f:
        [B_fluid, x_mesh, x_center, alpha_B] = pickle.load(f)
Delta_x = x_mesh[1:] - x_mesh[:-1]
x_center_extended = np.insert(x_center, 0, -x_center[0])
x_center_extended = np.append(x_center_extended, x_center[-1] + Delta_x[-1])
nu_anom_eff     = alpha_B * (phy_const.e/phy_const.m_e) * B_fluid

files       = glob.glob(ResultsData + "/MacroscopicVars_*.pkl")
filesSorted = sorted(files, key = lambda x: os.path.getmtime(x), reverse=True)
files.sort(key=os.path.getmtime)
print(ResultsData)
print(len(files))

files_1       = glob.glob(ResultsData_1 + "/MacroscopicVars_*.pkl")
filesSorted_1 = sorted(files_1, key = lambda x: os.path.getmtime(x), reverse=True)
files_1.sort(key=os.path.getmtime)


Current = np.zeros(np.shape(files)[0])
CurrentDensity = np.zeros(np.shape(files)[0])
Voltage = np.zeros(np.shape(files)[0])
time    = np.zeros(np.shape(files)[0])

for i_save, file in enumerate(files):
    try:
        with open(file, 'rb') as f:
            [t, P, U, P_LeftGhost, P_RightGhost, J, Efield, V] = pickle.load(f)
    except:
        with open(file, 'rb') as f:
                [t, P, U, P_Inlet, P_Outlet, J, V, B, x_center] = pickle.load(f)
        print("loaded new")
    
    # Save the current
    Current[i_save] = J
    CurrentDensity[i_save] = np.mean(P[1,:]*phy_const.e*(P[2,:] - P[4,:]))
    time[i_save]    = t
    Voltage[i_save] = V

In [ ]:
#####################################
#           Plot current
#####################################

f, ax = plt.subplots(figsize=(8,3))

ax.plot(time/1e-3, Current)
ax.set_xlabel('$t$ [ms]', fontsize=18)
ax.set_ylabel('$I_d$ [A]', fontsize=18)
ax_V=ax.twinx()
ax_V.plot(time/1e-3, Voltage,'g')
ax_V.set_ylabel('$V$ [V]', fontsize=18)
ax.grid(True)
plt.tight_layout()


In [ ]:
######################################
#    Plot the effective anomalous freq.
######################################
if Results == "/home/petronio/runs/fluid_results/charoy_code_base_N200/":
    f, ax = plt.subplots()
    ax_b = ax.twinx()
    ax.plot(x_center*100, nu_anom_eff/1e6, 'g+', label = "$\\nu_{anom}$")
    ax.set_ylim(0, 60)
    ax.set_xlim(0, 2.5)
    ax.set_ylabel('$\\nu_{a}$ [MHz]', fontsize=16)
    ax.set_xlabel('$x$ [cm]', fontsize=16)
    ax.yaxis.set_tick_params(which='both', size=5, width=1.5)
    ax.xaxis.set_tick_params(which='both', size=5, width=1.5)
    ax_b.yaxis.set_tick_params(which='both', size=5, width=1.5)
    ax_b.plot(x_center*100, B_fluid*1e4, 'r:')
    ax_b.set_ylabel('$B$ [Gauss]', color='r', fontsize=16)
    ax_b.set_ylim(0, 120)
    # plt.savefig(ResultsFigs+"/effective_anom_nu.png", bbox_inches='tight')
    # plt.close()

In [ ]:
print(len(files))
file = files[-1]
print(ResultsData)
try:
    with open(file, 'rb') as f:
        [t, P, U, P_LeftGhost, P_RightGhost, J, Efield_fluid, V] = pickle.load(f)
    ng_fluid = P[0,:]
    n_fluid = P[1,:]
    phi_fluid = compute_phi(Efield, Delta_x, J, V, 0.)
    vix_fluid = P[2,:]
    Te_fluid = P[3,:]
    v_bohm_fluid = np.sqrt(phy_const.e*P[3,:]/(131.293*phy_const.m_u))
    vex_fluid = P[4,:]
    vey_fluid = P[5,:]
    jd_tot_fluid = P[1,:]*phy_const.e*(P[2,:] - P[4,:])
    Rie_x_fluid = - cons.m_e * vex_fluid * nu_anom_eff * n_fluid
    Rie_y_fluid = - cons.m_e * vey_fluid * nu_anom_eff * n_fluid
    print("File 0 - LOADED")

except:
    with open(file, 'rb') as f:
        [t, P, U, P_Inlet, P_Outlet, J, V, B_fluid, x_center] = pickle.load(f)
    Delta_x = x_center[1:] - x_center[:-1]
    
    ResultConfig = Results+'/Configuration.cfg'
    configFile = ResultConfig
    config = configparser.ConfigParser()
    config.read(configFile)
    
    physicalParameters = config["Physical Parameters"]
    NBPOINTS = np.shape(x_center)[0]
    LX = float(physicalParameters["Length of axis"])  # length of Axis of the simulation
    x_mesh = np.linspace(0, LX, NBPOINTS + 1)  # Mesh in the interface
    x_center = (x_mesh[1:] + x_mesh[:-1])/2
    Delta_x  =  x_mesh[1:] - x_mesh[:-1]

    
    Efield_fluid = compute_E_with_az_motion(P, B, 0, 'None', 1, 1, 1, x_center, 0, 0, alpha_B, J, 0, new_model=True)
    
    phi = compute_phi(Efield_fluid, Delta_x, J, V, 0)
    
    PPP_0 = np.copy(P)
    
    ng = P[0, :]
    ni = P[1, :]
    vi = P[2, :]
    Te = P[3, :]
    ve = P[4, :]
    Ue_y = P[5, :]
    
    n_fluid = ni
    phi_fluid = phi
    vix_fluid = vi
    Te_fluid = Te
    v_bohm_fluid = np.sqrt(phy_const.e*P[3,:]/(131.293*phy_const.m_u))
    vex_fluid = ve
    vey_fluid = Ue_y
    vi_fluid = vi
    jd_tot_fluid = P[1,:]*phy_const.e*(P[2,:] - P[4,:])
    Rie_x_fluid = - cons.m_e * vex_fluid * 0 * n_fluid
    Rie_y_fluid = - cons.m_e * vey_fluid * 0 * n_fluid
    if Results == "/home/petronio/runs/fluid_results/half_gradPxy_emp_term_add_6":
        try:
            empirical_term = np.loadtxt('/home/petronio/runs/fluid_results/empirical_term_add_4.txt')
            empirical_term_interp = np.interp(x_center, empirical_term[:, 0]/100, empirical_term[:, 1])
            Rie_y_fluid = empirical_term_interp 
            Rie_x_fluid = Rie_x_fluid*0
        except:
            print("No empirical term file found.")


# Plotting
# titles_fluid = ["$E_x$ [V/m]", "$n_e$ [m$^{-3}$]", r"$e\langle \delta n \delta E_y \rangle_y$ [J m$^{-4}$]", r"$e\langle \delta n \delta E_x \rangle_y$ [J m$^{-4}$]", "$v_{e,y}$ [m/s]", "$v_{e,x}$ [m/s]", "$T_{e,x}$ [eV]", "$\Phi$ [V]", "$B [G]$"]
data_fluid = [Efield_fluid/1e3, n_fluid, Rie_y_fluid, Rie_x_fluid, vey_fluid/1E6, vex_fluid/1E3, Te_fluid, phi_fluid, B_fluid, vix_fluid/1e3]

# titles_fluid = ["$E_x$ [V/m]", "$n_e$ [m$^{-3}$]", "$v_{e,y}$ [m/s]", "$v_{e,x}$ [m/s]", "$T_{e}$ [eV]", "$\Phi$ [V]", "$B [G]$"]
data_fluid = [Efield_fluid/1e3, n_fluid/1e17, vey_fluid/1E6, vex_fluid/1E3, Te_fluid, phi_fluid, B_fluid, vix_fluid/1e3]

data_fluid_new = [Efield_fluid/1e3, phi_fluid, n_fluid/1e17,  Te_fluid,  vix_fluid/1e3, vey_fluid/1E6, vex_fluid/1E3, B_fluid]


In [ ]:
file = files_1[-1]

print(ResultsData_1)

try:
    with open(file, 'rb') as f:
        [t, P, U, P_LeftGhost, P_RightGhost, J, Efield_fluid, V] = pickle.load(f)
    n_fluid = P[1,:]
    phi_fluid = compute_phi(Efield, Delta_x, J, V, 0.)
    vix_fluid = P[2,:]
    Te_fluid = P[3,:]
    v_bohm_fluid = np.sqrt(phy_const.e*P[3,:]/(131.293*phy_const.m_u))
    vex_fluid = P[4,:]
    vey_fluid = P[5,:]
    jd_tot_fluid = P[1,:]*phy_const.e*(P[2,:] - P[4,:])
    Rie_x_fluid = - cons.m_e * vex_fluid * nu_anom_eff * n_fluid
    Rie_y_fluid = - cons.m_e * vey_fluid * nu_anom_eff * n_fluid
    print("File 1 - LOADED")
except:
    with open(file, 'rb') as f:
        [t, P, U, P_Inlet, P_Outlet, J, V, B_fluid, x_center] = pickle.load(f)
    Delta_x = x_center[1:] - x_center[:-1]
    
    ResultConfig = Results+'/Configuration.cfg'
    configFile = ResultConfig
    config = configparser.ConfigParser()
    config.read(configFile)
    
    physicalParameters = config["Physical Parameters"]
    NBPOINTS = np.shape(x_center)[0]
    LX = float(physicalParameters["Length of axis"])  # length of Axis of the simulation
    x_mesh = np.linspace(0, LX, NBPOINTS + 1)  # Mesh in the interface
    x_center = (x_mesh[1:] + x_mesh[:-1])/2
    Delta_x  =  x_mesh[1:] - x_mesh[:-1]

    
    Efield_fluid = compute_E_with_az_motion(P, B_fluid, 0, 'None', 1, 1, 1, x_center, 0, 0, alpha_B, J, 0, new_model=True)
    
    phi = compute_phi(Efield_fluid, Delta_x, J, V, 0)
    
    PPP_1 = np.copy(P)
    
    ng = P[0, :]
    ni = P[1, :]
    vi = P[2, :]
    Te = P[3, :]
    ve = P[4, :]
    Ue_y = P[5, :]
    
    n_fluid = ni
    phi_fluid = phi
    vix_fluid = vi
    Te_fluid = Te
    v_bohm_fluid = np.sqrt(phy_const.e*P[3,:]/(131.293*phy_const.m_u))
    vex_fluid = ve
    vey_fluid = Ue_y
    vi_fluid = vi
    jd_tot_fluid = P[1,:]*phy_const.e*(P[2,:] - P[4,:])
    Rie_x_fluid = - cons.m_e * vex_fluid * 0 * n_fluid
    Rie_y_fluid = - cons.m_e * vey_fluid * 0 * n_fluid
    if Results == "/home/petronio/runs/fluid_results/half_gradPxy_emp_term_add_6":
        try:
            empirical_term = np.loadtxt('/home/petronio/runs/fluid_results/empirical_term_add_4.txt')
            empirical_term_interp = np.interp(x_center, empirical_term[:, 0]/100, empirical_term[:, 1])
            Rie_y_fluid = empirical_term_interp 
            Rie_x_fluid = Rie_x_fluid*0
        except:
            print("No empirical term file found.")


# Plotting
titles_fluid = ["$E_x$ [V/m]", "$n_e$ [m$^{-3}$]", r"$e\langle \delta n \delta E_y \rangle_y$ [J m$^{-4}$]", r"$e\langle \delta n \delta E_x \rangle_y$ [J m$^{-4}$]", "$v_{e,y}$ [m/s]", "$v_{e,x}$ [m/s]", "$T_{e,x}$ [eV]", "$\Phi$ [V]", "$B [G]$"]
# data_fluid = [Efield_fluid/1e3, n_fluid, Rie_y_fluid, Rie_x_fluid, vey_fluid/1E6, vex_fluid/1E3, Te_fluid, phi_fluid, B_fluid, vix_fluid/1e3]

titles_fluid = ["$E_x$ [V/m]", "$n_e$ [m$^{-3}$]", "$v_{e,y}$ [m/s]", "$v_{e,x}$ [m/s]", "$T_{e}$ [eV]", "$\Phi$ [V]", "$B [G]$"]
data_fluid_1 = [Efield_fluid/1e3, n_fluid/1e17, vey_fluid/1E6, vex_fluid/1E3, Te_fluid, phi_fluid, B_fluid, vix_fluid/1e3]

data_fluid_new = [Efield_fluid/1e3, phi_fluid, n_fluid/1e17,  Te_fluid,  vix_fluid/1e3, vey_fluid/1E6, vex_fluid/1E3, B_fluid]


In [ ]:
index = 0

titles_PIC = ["$E_x$ [kV/m]", "$n_e$ [$10^{17}$m$^{-3}$]","$v_{e,y}$ [$10^6$m/s]", "$v_{e,x}$ [km/s]", "$T_{e,x}$ [eV]", "$\Phi$ [V]", "$B$ [T]", "$v_{i,x}$ [km/s]", "Te_tot [eV]"]


for title, dfluid, dfluid_1 in zip(titles_PIC, data_fluid, data_fluid_1):
    plt.figure()
    ax = plt.subplot(111)
    ax.plot(x_center*100, dfluid, linestyle=":", label="fluid", marker = "v")
    ax.plot(x_center*100, dfluid_1, linestyle=":", label="fluid ref")
    ax.set_ylabel(title, fontsize=16)
    ax.set_xlabel("$x$ [cm]", fontsize=16)
    ax.grid(False)
    ax.set_xlim(0,2.5)
    ax.legend()
    if index == 0:
        ax.set_ylim(-20, 60)
    elif index == 3:
        ax.set_ylim(-50, 10)
    elif index == 7:
        ax.set_ylim(-5, 20)
    index += 1

    plt.tight_layout()

    plt.savefig(image_folder + label_res + "_"+ str(index) + "_" + ".png", dpi = 300, transparent=True)
    # plt.close()

In [ ]:
plt.figure()
plt.plot(ng_fluid)